# Imports

In [2]:
import os
import sys

project_root = "Incremental-Learning-MycroftMinds"
cwd = os.path.abspath(os.getcwd())
while True:
    if os.path.basename(cwd) == project_root:
        if cwd not in sys.path:
            sys.path.insert(0, cwd)
        break
    parent = os.path.dirname(cwd)
    if parent == cwd:
        raise FileNotFoundError(f"Project root '{project_root}' not found in any parent directory.")
    cwd = parent

In [3]:
import time

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from river.forest import ARFRegressor
from river.tree import HoeffdingTreeRegressor
from src.bagging import ARFRegressorFODT
from src.trees import fastodt
from src.boosting import IncubationBoost
from src.ensemble_methods import elasticGBDT
from src.evaluation.baseline import PreviousValueRegressor

from src.data.io import load_dataset
from src.evaluation.tuning import grid_search_tuning
from src.data import dataloader
from src.utils.logging import load_checkpoint_results, list_log_files, view_log_tail

# Make the splits (if not already done)

In [ ]:
lag_args = {
    #'lag_5m':'5M',
    'lag_1h':'1H',
    'lag_24h':'24H',
    }

data_dir = '../../data/first_batch'

columns_to_add_lag = [
            'L1_act_power', 'L1_app_power', 'L1_current', 'L1_react_power', 'L1_voltage',
            'L2_act_power', 'L2_app_power', 'L2_current', 'L2_react_power', 'L2_voltage',
            'L3_act_power', 'L3_app_power', 'L3_current', 'L3_react_power', 'L3_voltage',
            'grid_freq', 'powerfactor1', 'powerfactor2', 'powerfactor3'
        ]
device_list = ['df_TRB145-4',
               #'df_TRB145-5',
               ]

# target
target_feature = 'L2_act_power'

train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data for faster future loading
    intermediate_prefix='dev_4_hourly_wfv',  # Prefix for saved files
    time_agg='1H',  # Time aggregation frequency
    new_data=True,
    fill_missing=True
)

# Load the data

In [4]:
HOURLY = True
DATA_FREQUENCY = 'H' if HOURLY else '5min'

#data_dir = '/content/drive/MyDrive/extracted_files/first_batch'
data_dir = '../../data/first_batch'
# Measure loading time
start_time = time.time()

# Load data
if HOURLY:
    train, valid, test = load_dataset(data_dir=data_dir, prefix='dev_4_1h_wfv', use_numpy=True)
    features = ['time', 'device_id', 'hour', 'weekday', 'month',
                 'L1_act_power_lag_1h', 'L1_act_power_lag_24h', 'L1_app_power_lag_1h', 'L1_app_power_lag_24h',
                 'L1_current_lag_1h', 'L1_current_lag_24h', 'L1_react_power_lag_1h', 'L1_react_power_lag_24h',
                 'L1_voltage_lag_1h', 'L1_voltage_lag_24h',
                 'L2_act_power_lag_1h', 'L2_act_power_lag_24h', 'L2_app_power_lag_1h', 'L2_app_power_lag_24h',
                 'L2_current_lag_1h', 'L2_current_lag_24h', 'L2_react_power_lag_1h', 'L2_react_power_lag_24h',
                 'L2_voltage_lag_1h', 'L2_voltage_lag_24h',
                 'L3_act_power_lag_1h', 'L3_act_power_lag_24h', 'L3_app_power_lag_1h', 'L3_app_power_lag_24h',
                 'L3_current_lag_1h', 'L3_current_lag_24h', 'L3_react_power_lag_1h', 'L3_react_power_lag_24h',
                 'L3_voltage_lag_1h', 'L3_voltage_lag_24h',
                 'grid_freq_lag_1h', 'grid_freq_lag_24h',
                 'powerfactor1_lag_1h', 'powerfactor1_lag_24h',
                 'powerfactor2_lag_1h', 'powerfactor2_lag_24h',
                 'powerfactor3_lag_1h', 'powerfactor3_lag_24h']
else:
    train, valid, test = load_dataset(data_dir=data_dir, prefix='8_dev_5_min_wfv', use_numpy=True)
    # copy features from dataloader printouts
    features = []

loading_time = time.time() - start_time
print(f"Data loaded in {loading_time/60:.2f} minutes")

[INFO] Loading train data...
[INFO] Train data loaded: 7 samples
[INFO] Loading valid data...
[INFO] Valid data loaded: 7 samples
[INFO] Loading test data...
[INFO] Test data loaded: 2 samples
Data loaded in 0.01 minutes


# Hyperparameter tuning

The enhanced grid search will automatically handle logging and checkpoint creation with custom file prefixes:
- Creates log directories automatically with sensible defaults
- Uses custom file prefixes (e.g., 'dev_4_wfv') for organized file naming
- Logs all parameter combinations and results to timestamped files
- Saves intermediate results to prevent data loss
- Provides detailed progress tracking

File naming pattern: `timestamp_prefix_modelname_tuning.txt` and `prefix_modelname_results.pkl`

## Configuration

Set up experiment configuration and file naming.

In [5]:
# Experiment configuration
EXPERIMENT_PREFIX = "dev_4_1h_wfv"  # This will be used in all result filenames
BATCH_MODELS_LOG_PREFIX = "batch_models"
INCREMENTAL_MODELS_LOG_PREFIX = "incremental_models_v2"

print(f"Experiment prefix: {EXPERIMENT_PREFIX}")
print(f"Batch models will be saved to: ../results/{BATCH_MODELS_LOG_PREFIX}_results/")
print(f"Incremental models will be saved to: ../results/{INCREMENTAL_MODELS_LOG_PREFIX}_results/")
print(f"Files will be named with pattern: {EXPERIMENT_PREFIX}_ModelName_results.pkl")

Experiment prefix: dev_4_1h_wfv
Batch models will be saved to: ../results/batch_models_results/
Incremental models will be saved to: ../results/incremental_models_v2_results/
Files will be named with pattern: dev_4_1h_wfv_ModelName_results.pkl


## Batch-Learning Models

### GBDT

In [ ]:
# GBDT hyperparameter tuning
gbdt = GradientBoostingRegressor

param_grid_gbdt = {
    'n_estimators': [25, 50, 100, 150, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'max_depth': [3, 7, 12, 15],
    'subsample': [0.6, 0.8, 1.0],
    'random_state': [42]
}

print("Starting GBDT hyperparameter tuning...")
print(f"Total combinations: {np.prod([len(v) for v in param_grid_gbdt.values()])}")

best_params_gbdt, best_scores_gbdt = grid_search_tuning(
    param_grid_gbdt, 
    gbdt, 
    train, 
    valid, 
    aggregation='hourly',
    data_freq=DATA_FREQUENCY,
    update_freq='14D',  # Batch learning update frequency
    log_prefix=BATCH_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX,
    save_intermediate=True
)

In [ ]:
print("GBDT Best Parameters:")
print(best_params_gbdt)
print("\nGBDT Best Scores:")
print(best_scores_gbdt)

### Random Forest

In [ ]:
# Random Forest hyperparameter tuning
rf = RandomForestRegressor

param_grid_rf = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth': [3, 5, 7, 10, 15, 20],
    'max_features': [0.3, 0.5, 0.7, 1.0],
    'random_state': [42],
    'n_jobs': [-1]
}

print("Starting Random Forest hyperparameter tuning...")
best_params_rf, best_scores_rf = grid_search_tuning(
    param_grid_rf, 
    rf, 
    train, 
    valid, 
    aggregation='hourly',
    data_freq=DATA_FREQUENCY,
    update_freq='14D',  # Batch learning update frequency
    log_prefix=BATCH_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

In [8]:
print("Random Forest Best Parameters:")
for key, value in best_params_rf.items():
    print(f"{key}: {value}")

print("\nRandom Forest Best Scores:")
for key, value in best_scores_rf.items():
    print(f"{key}: {value}")

Random Forest Best Parameters:
rmse: {'n_estimators': 150, 'max_depth': 15, 'max_features': 0.5, 'random_state': 42, 'n_jobs': -1}
mae: {'n_estimators': 50, 'max_depth': 7, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}
mape: {'n_estimators': 50, 'max_depth': 7, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}
dir_accuracy: {'n_estimators': 200, 'max_depth': 20, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}
first_diff_corr: {'n_estimators': 300, 'max_depth': 5, 'max_features': 0.3, 'random_state': 42, 'n_jobs': -1}
spikes_f1: {'n_estimators': 50, 'max_depth': 5, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}
drops_f1: {'n_estimators': 200, 'max_depth': 20, 'max_features': 0.3, 'random_state': 42, 'n_jobs': -1}

Random Forest Best Scores:
rmse: 0.23034632600506094
mae: 0.14018759656971155
mape: 0.27282002820412193
dir_accuracy: 0.6355920366570883
first_diff_corr: 0.4450692328467722
spikes_f1: 0.2041448158210653
drops_f1: 0.1425958719191802


## Incremental Learning Models

### HoeffdingTree

In [ ]:
update_freq = "1h"

hoef = HoeffdingTreeRegressor

param_grid = {
    'max_depth': [3, 5, 10, 15, 20],
    'grace_period': [0, 10, 24, 48, 69, 96],
    'tau': [0.001, 0.005, 0.01, 0.03, 0.05, 0.1],
    'delta': [0.001, 0.005, 0.01, 0.05, 0.1],
    #'merit_preprune': [True, False]
}

print("Starting HoeffdingTree hyperparameter tuning...")
best_params_hf, best_scores_hf = grid_search_tuning(
    param_grid, 
    hoef, 
    train, 
    valid, 
    update_freq=update_freq,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

In [7]:
print("HoeffdingTree Best Parameters:")
print(best_params_hf)
print("\nHoeffdingTree Best Scores:")
print(best_scores_hf)

HoeffdingTree Best Parameters:
{'rmse': {'max_depth': 10, 'grace_period': 48, 'tau': 0.1, 'delta': 0.1}, 'mae': {'max_depth': 15, 'grace_period': 48, 'tau': 0.1, 'delta': 0.1}, 'mape': {'max_depth': 15, 'grace_period': 48, 'tau': 0.1, 'delta': 0.1}, 'dir_accuracy': {'max_depth': 5, 'grace_period': 0, 'tau': 0.1, 'delta': 0.005}, 'first_diff_corr': {'max_depth': 3, 'grace_period': 0, 'tau': 0.1, 'delta': 0.05}, 'spikes_f1': {'max_depth': 3, 'grace_period': 24, 'tau': 0.001, 'delta': 0.05}, 'drops_f1': {'max_depth': 10, 'grace_period': 24, 'tau': 0.1, 'delta': 0.005}}

HoeffdingTree Best Scores:
{'rmse': 0.2584388053746239, 'mae': 0.1618406777890497, 'mape': 0.3020870092444711, 'dir_accuracy': 0.5417023198165519, 'first_diff_corr': 0.4167744519934921, 'spikes_f1': 0.18787111743922377, 'drops_f1': 0.10620447763304908}


### ARF (Hoeffding Tree)

In [ ]:
param_grid = {
    'n_models': [5, 10, 20, 40, 60, 80, 100],
    'max_features': [0.1, 0.3, 0.5, 0.7, 1.0],
    'max_depth': [5, 7, 10, 15, 25, 30, 50, 70, 100],
    'aggregation_method': ['mean', 'median'],
    'disable_weighted_vote': [True, False],
    'drift_detector': [None],
    'warning_detector': [None],
    'tau': [0.001, 0.005, 0.01, 0.03, 0.05, 0.1],
    'delta': [0.001, 0.005, 0.01, 0.05, 0.1],
    'grace_period': [0, 10, 24, 35, 48, 69, 96],
}

arf_hf = ARFRegressor

print("Starting ARF (Hoeffding Tree) hyperparameter tuning...")
best_params_arf_hf, best_scores_arf_hf = grid_search_tuning(
    param_grid, 
    arf_hf, 
    train, 
    valid, 
    update_freq=update_freq,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

In [ ]:
print("ARF (Hoeffding Tree) Best Parameters:")
print(best_params_arf_hf)
print("\nARF (Hoeffding Tree) Best Scores:")
print(best_scores_arf_hf)

### FastODT

In [6]:
update_freq = "1h"  # for incremental models put the same as the data frequency to recreate real world scenario

fodt = fastodt.FastODT

param_grid = {
    'depth': [5, 10, 15, 30, 50, 100],
    'grace_period': [0, 10, 24, 48, 96, 168],
    'grace_period_depth': [-1, 0, 1],
    'split_confidence': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5],
    'max_feature': [0.3, 0.5, 0.7, 1.0],
    'tie_threshold': [0.01, 0.05],
    'bin_type': ['mean'],
    'split_criterion': ['vr_rel']
}

print("Starting FastODT hyperparameter tuning...")
best_params_fastODT, best_scores_fastODT = grid_search_tuning(
    param_grid, 
    fodt, 
    train, 
    valid, 
    update_freq=update_freq,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

Starting FastODT hyperparameter tuning...


Grid Search Progress:   0%|          | 0/5184 [00:00<?, ?it/s]

Processing splits:   0%|          | 0/7 [00:00<?, ?it/s]

Processing devices in split 0:   0%|          | 0/1 [00:00<?, ?it/s]

Processing devices in split 1:   0%|          | 0/1 [00:00<?, ?it/s]

Processing devices in split 2:   0%|          | 0/1 [00:00<?, ?it/s]

... (41,991 outputs left out)

Processing devices in split 4:   0%|          | 0/1 [00:00<?, ?it/s]

Processing devices in split 5:   0%|          | 0/1 [00:00<?, ?it/s]

Processing devices in split 6:   0%|          | 0/1 [00:00<?, ?it/s]

Results saved to ../results/incremental_models_v2_results\dev_4_1h_wfv_FastODT_results.pkl


In [7]:
print("FastODT Best Parameters:")
for key, value in best_params_fastODT.items():
    print(f"{key}: {value}")
print("\nFastODT Best Scores:")
for key, value in best_scores_fastODT.items():
    print(f"{key}: {value}")

FastODT Best Parameters:
rmse: {'depth': 100, 'grace_period': 24, 'grace_period_depth': -1, 'split_confidence': 0.5, 'max_feature': 0.7, 'tie_threshold': 0.05, 'bin_type': 'mean', 'split_criterion': 'vr_rel'}
mae: {'depth': 50, 'grace_period': 24, 'grace_period_depth': 0, 'split_confidence': 0.005, 'max_feature': 0.3, 'tie_threshold': 0.05, 'bin_type': 'mean', 'split_criterion': 'vr_rel'}
mape: {'depth': 50, 'grace_period': 24, 'grace_period_depth': 0, 'split_confidence': 0.005, 'max_feature': 0.3, 'tie_threshold': 0.05, 'bin_type': 'mean', 'split_criterion': 'vr_rel'}
dir_accuracy: {'depth': 50, 'grace_period': 24, 'grace_period_depth': 0, 'split_confidence': 0.5, 'max_feature': 1.0, 'tie_threshold': 0.05, 'bin_type': 'mean', 'split_criterion': 'vr_rel'}
first_diff_corr: {'depth': 5, 'grace_period': 96, 'grace_period_depth': 1, 'split_confidence': 0.005, 'max_feature': 1.0, 'tie_threshold': 0.05, 'bin_type': 'mean', 'split_criterion': 'vr_rel'}
spikes_f1: {'depth': 5, 'grace_period': 

### ARF (FastODT)

In [ ]:
update_freq = "1h"

arf = ARFRegressorFODT

param_grid = {
    'n_models': [5, 10, 30, 60, 100],
    'max_features': [0.3, 0.5, 0.7, 1.0],
    'max_depth': [5, 10, 25, 50, 100],
    'aggregation_method': ['mean', 'median'],
    'disable_weighted_vote': [True, False],
    'drift_detector': [None],
    'warning_detector': [None],
    'eagerness': [0, 100, 500, 1000],
    #'tau': [0.001, 0.005, 0.01, 0.03, 0.05, 0.1],
    'delta': [0.001, 0.005, 0.01, 0.05, 0.1],
    'grace_period': [0, 10, 24, 48],
    'grace_period_depth': [-1, 0, 1]
}

print("Starting ARF (FastODT) hyperparameter tuning...")
best_params_arf, best_scores_arf = grid_search_tuning(
    param_grid, 
    arf, 
    train, 
    valid, 
    update_freq=update_freq,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

In [ ]:
print("ARF (FastODT) Best Parameters:")
print(best_params_arf)
print("\nARF (FastODT) Best Scores:")
print(best_scores_arf)

{'rmse': {'n_models': np.int64(20), 'max_depth': np.int64(250), 'eagerness': 200, 'delta': np.float64(0.001), 'aggregation_method': 'mean', 'tau': 0.01, 'grace_period': 0}, 'mae': {'n_models': np.int64(30), 'max_depth': np.int64(250), 'eagerness': 200, 'delta': np.float64(0.001), 'aggregation_method': 'mean', 'tau': 0.01, 'grace_period': 0}, 'mape': {'n_models': np.int64(10), 'max_depth': np.int64(50), 'eagerness': 200, 'delta': np.float64(0.005), 'aggregation_method': 'mean', 'tau': 0.01, 'grace_period': 0}}
{'rmse': 0.389549305491319, 'mae': 0.2342591609672969, 'mape': 0.5789888521014152}


### Incubation boost

In [ ]:
import itertools

update_freq = "1h"  # for incremental models put the same as the data frequency to recreate real world scenario:

incb = IncubationBoost

fodt_param_grid = {
    'depth': [5, 10, 25, 50, 70, 100],
    'grace_period': [0, 10, 24, 35, 48, 96],
    'split_confidence': [0.001, 0.005, 0.01, 0.05, 0.1],
    'option':['hf'],
    'max_feature': [0.3, 0.5, 0.7, 1.0],
    'eagerness': [0, 100, 500, 1000],
    'grace_period_depth': [-1, 0, 1],
}
# Generate all combinations of fodt_param_grid
baselLearnerOptions = list(itertools.product(
    fodt_param_grid['depth'],
    fodt_param_grid['grace_period'],
    fodt_param_grid['split_confidence'],
    fodt_param_grid['option'],
    fodt_param_grid['max_feature'],
    fodt_param_grid['eagerness'],
    fodt_param_grid['grace_period_depth']
))

param_grid = {
    'baseLearner': [fastodt.FastODT],
    'length': [5, 10, 25, 50, 70],
    'incubationMode': ["DEFAULT", "BREAK", "SHIFT_LEFT", "SHIFT_RIGHT"],
    'baseLearnerOptions': baselLearnerOptions,
    'verbose':[False]
}

print("Starting Incubation Boost hyperparameter tuning...")
best_params_inc, best_scores_inc = grid_search_tuning(
    param_grid,
    incb, 
    train, 
    valid, 
    update_freq=update_freq,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

In [ ]:
import itertools

update_freq = "1h"  # for incremental models put the same as the data frequency to recreate real world scenario:

incb = IncubationBoost

fodt_param_grid = {
    'depth': [15, 30, 50, 70],
    'grace_period': [10],
    'split_confidence': [0.005],
    'option': ['hf'],
    'max_feature': [1.0],
    'eagerness': [1000],
    'grace_period_depth': [0],
}
# Generate all combinations of fodt_param_grid
baselLearnerOptions = list(itertools.product(
    fodt_param_grid['depth'],
    fodt_param_grid['grace_period'],
    fodt_param_grid['split_confidence'],
    fodt_param_grid['option'],
    fodt_param_grid['max_feature'],
    fodt_param_grid['eagerness'],
    fodt_param_grid['grace_period_depth']
))

param_grid = {
    'baseLearner': [fastodt.FastODT],
    'length': [50, 70, 100],
    'learning_rate': [1.0, 0.05, 0.5],
    'incubationMode': ["DEFAULT"],
    'baseLearnerOptions': baselLearnerOptions,
    'verbose':[False]
}

print("Starting Incubation Boost hyperparameter tuning...")
best_params_inc, best_scores_inc = grid_search_tuning(
    param_grid,
    incb, 
    train, 
    valid, 
    update_freq=update_freq,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)


In [9]:
print("Incubation Boost Best Parameters:")
for key, value in best_params_inc.items():
    print(f"{key}: {value}")

print("\nIncubation Boost Best Scores:")
for key, value in best_scores_inc.items():
    print(f"{key}: {value}")

Incubation Boost Best Parameters:
rmse: {'baseLearner': <class 'src.trees.fastodt.FastODT'>, 'length': 70, 'learning_rate': 1.0, 'incubationMode': 'DEFAULT', 'baseLearnerOptions': (15, 10, 0.005, 'hf', 1.0, 1000, 0), 'verbose': False}
mae: {'baseLearner': <class 'src.trees.fastodt.FastODT'>, 'length': 70, 'learning_rate': 1.0, 'incubationMode': 'DEFAULT', 'baseLearnerOptions': (15, 10, 0.005, 'hf', 1.0, 1000, 0), 'verbose': False}
mape: {'baseLearner': <class 'src.trees.fastodt.FastODT'>, 'length': 50, 'learning_rate': 1.0, 'incubationMode': 'DEFAULT', 'baseLearnerOptions': (70, 10, 0.005, 'hf', 1.0, 1000, 0), 'verbose': False}
dir_accuracy: {'baseLearner': <class 'src.trees.fastodt.FastODT'>, 'length': 50, 'learning_rate': 1.0, 'incubationMode': 'DEFAULT', 'baseLearnerOptions': (50, 10, 0.005, 'hf', 1.0, 1000, 0), 'verbose': False}
first_diff_corr: {'baseLearner': <class 'src.trees.fastodt.FastODT'>, 'length': 50, 'learning_rate': 1.0, 'incubationMode': 'DEFAULT', 'baseLearnerOptions'

# Results Summary

## All Best Hyperparameters

In [ ]:
def create_metric_param_table(all_best_params):
    """
    Creates a DataFrame where rows are model names, columns are metrics (rmse, mae, mape),
    and each cell contains the corresponding best param dict for that model and metric.

    Args:
        all_best_params (dict): {model_name: {'rmse': param_dict, 'mae': ..., 'mape': ...}}

    Returns:
        pd.DataFrame: DataFrame of shape (n_models x 7) with best params as values
    """
    metrics = ['rmse', 'mae', 'mape', 'dir_accuracy', 'first_diff_corr', 'spikes_f1', 'drops_f1']
    data = {}

    for model_name, metric_dict in all_best_params.items():
        row = {}
        for metric in metrics:
            row[metric] = metric_dict.get(metric, None)
        data[model_name] = row

    df = pd.DataFrame.from_dict(data, orient='index')
    df.index.name = 'model'
    return df

In [ ]:
all_best_params = {
    'GBDT': best_params_gbdt,
    'RandomForest': best_params_rf,
    'HoeffdingTreeRegressor': best_params_hf,
    'ARF (Hoeffding Tree)': best_params_arf_hf,
    'ARF (FastODT)': best_params_arf,
    'FastODT': best_params_fastODT,
    'IncubationBoost': best_params_inc
}

metric_param_table = create_metric_param_table(all_best_params)
print("Best Hyperparameters by Model and Metric:")
display(metric_param_table)

## Best Scores Summary

In [ ]:
def create_metric_scores_table(all_best_scores):
    """
    Creates a DataFrame where rows are model names, columns are metrics,
    and each cell contains the best score for that model and metric.
    """
    metrics = ['rmse', 'mae', 'mape', 'dir_accuracy', 'first_diff_corr', 'spikes_f1', 'drops_f1']
    data = {}

    for model_name, metric_dict in all_best_scores.items():
        row = {}
        for metric in metrics:
            score = metric_dict.get(metric, None)
            if score is not None:
                # Format percentage metrics
                if metric in ['mape', 'dir_accuracy', 'first_diff_corr', 'spikes_f1', 'drops_f1']:
                    row[metric] = f"{score*100:.3f}%" if score != float('inf') and score != float('-inf') else "N/A"
                else:
                    row[metric] = f"{score:.3f}" if score != float('inf') else "N/A"
            else:
                row[metric] = "N/A"
        data[model_name] = row

    df = pd.DataFrame.from_dict(data, orient='index')
    df.index.name = 'model'
    return df

all_best_scores = {
    'GBDT': best_scores_gbdt,
    'RandomForest': best_scores_rf,
    'HoeffdingTreeRegressor': best_scores_hf,
    'ARF (Hoeffding Tree)': best_scores_arf_hf,
    'ARF (FastODT)': best_scores_arf,
    'FastODT': best_scores_fastODT,
    'IncubationBoost': best_scores_inc
}

scores_table = create_metric_scores_table(all_best_scores)
print("Best Scores by Model and Metric:")
display(scores_table)